In [1]:
import pandas as pd

file_path = "dataset/1742574481558_Dataset-ME-2025.xlsx"  # ganti dengan path ke file kamu

# --- 1. DC Sheet: Tampilkan Area dan jumlah baris per Area ---
df_dc = pd.read_excel(file_path, sheet_name="DC")
area_counts = df_dc['Area'].value_counts().reset_index()
area_counts.columns = ['Area', 'Jumlah Baris']
# print("Jumlah baris per Area di sheet 'DC':")
# print(area_counts)

In [2]:
df_dc_demand = df_dc[['Area', 'DC ID', 'Distributor Area', 'Type','Sent From', 'Latitude', 'Longitude',
                      'Monthly Demand (Average FY2021) - Carton', 
                      'Max Demand - FY2021 - Carton']]

df_dc_demand['Monthly Demand (Average FY2021) - Carton'] = df_dc_demand['Monthly Demand (Average FY2021) - Carton'].round(0)
df_dc_demand['Max Demand - FY2021 - Carton'] = df_dc_demand['Max Demand - FY2021 - Carton'].round(0)


df_dc_demand.head()

C:\Users\Bara\AppData\Local\Temp\ipykernel_33628\1304387991.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dc_demand['Monthly Demand (Average FY2021) - Carton'] = df_dc_demand['Monthly Demand (Average FY2021) - Carton'].round(0)
C:\Users\Bara\AppData\Local\Temp\ipykernel_33628\1304387991.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dc_demand['Max Demand - FY2021 - Carton'] = df_dc_demand['Max Demand - FY2021 - Carton'].round(0)


,Area,DC ID,Distributor Area,Type,Sent From,Latitude,Longitude,Monthly Demand (Average FY2021) - Carton,Max Demand - FY2021 - Carton
0,Ampah - Teweh,DC001,Kalimantan,Depo,NaN,-0.935414,114.901123,NaN,NaN
1,Amurang,DC002,BCP Outer,Depo,Manado,1.255330,124.631870,NaN,NaN
2,Balikpapan,DC037,Kalimantan,Direct DC,SDC,-1.249963,116.868785,10546.0,12817.0
3,Banjarmasin,DC038,Kalimantan,Direct DC,SDC,-3.330249,114.595582,24354.0,38897.0
4,Barabai,DC003,Kalimantan,Depo,NaN,-2.581936,115.390569,NaN,NaN


In [3]:
import pandas as pd

# 1. Baca sheet Resources dengan header di baris ke-3
df_region = pd.read_excel(file_path, sheet_name='Resources', header=3, usecols=['REGION', 'BRANCH'])

# 2. Bersihkan dan lowercase
df_region['BRANCH'] = df_region['BRANCH'].str.strip().str.lower()
df_region['REGION'] = df_region['REGION'].str.strip()

# Ganti 'bali' menjadi 'denpasar' di df_region
df_region['BRANCH'] = df_region['BRANCH'].replace('bali', 'denpasar')

# 3. Standardisasi df_dc_demand
df_dc_demand['Area'] = df_dc_demand['Area'].str.strip().str.lower()

# 4. Join berdasarkan Area dan Branch
df_dc_enriched = df_dc_demand.merge(
    df_region.drop_duplicates(),
    how='left',
    left_on='Area',
    right_on='BRANCH'
)

# 5. Drop kolom BRANCH (optional)
df_dc_enriched.drop(columns=['BRANCH'], inplace=True)

# 6. Tangani Area yang REGION-nya masih NaN → isi "Unknown"
# 6. Tangani NaN di REGION:
# Jika Type == 'Depo', isi REGION = 'Depo'
# Jika tidak, isi REGION = 'Unknown'
df_dc_enriched['REGION'] = df_dc_enriched.apply(
    lambda row: 'Depo' if pd.isna(row['REGION']) and row['Type'].lower() == 'depo' else (
        'Unknown' if pd.isna(row['REGION']) else row['REGION']
    ),
    axis=1
)

# 7. Contoh hasil
print(df_dc_enriched[['DC ID', 'Area', 'REGION']].sample(10))


    DC ID             Area    REGION
13  DC040          bontang   Unknown
18  DC010             gowa      Depo
52  DC027          takalar      Depo
37  DC018          pangkep      Depo
54  DC028          tomohon      Depo
40  DC020   penajam-grogot      Depo
30  DC015       mangkutana      Depo
14  DC008  bontang-sangata      Depo
28  DC052           mamuju  Sulawesi
7   DC046          bau-bau  Sulawesi


C:\Users\Bara\AppData\Local\Temp\ipykernel_33628\172746499.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dc_demand['Area'] = df_dc_demand['Area'].str.strip().str.lower()


In [4]:
newxls = "dataset/Dataset Bersih Lengkap.xlsx"
df_customer = pd.read_excel(newxls, sheet_name="Customer Final")

df_customer.head()
# Ambil kolom-kolom yang diperlukan dari df_customer
df_customer_filtered = df_customer[[
    'Customer ID',
    'Sub-District Area',
    'Latitude',
    'Longitude',
    'Avg Monthly Demand (box)'
]].copy()
df_customer_filtered.head()


FileNotFoundError: [Errno 2] No such file or directory: 'dataset/Dataset Bersih Lengkap.xlsx'

In [ ]:
import pandas as pd
from geopy.distance import geodesic

# PARAMETER
MAX_DELIVERY_DISTANCE = 100000  # km, batas maksimal jarak realistis

# Step 1: Filter DC (tanpa Pabrik / SDC)
df_dc_deliver = df_dc_enriched[df_dc_enriched['Type'] != 'Pabrik'].copy()
df_dc_deliver['Remaining Capacity'] = df_dc_deliver['Max Demand - FY2021 - Carton'].copy()

# Step 2: Sampling customer (bisa 117k++ nanti kalau sudah stabil)
# df_customer_sample = df_customer.sample(n=50000, random_state=100)

# Step 3: Alokasi
allocation_result = []

for idx, cust in df_customer_filtered.iterrows():
    cust_coord = (cust['Latitude'], cust['Longitude'])
    cust_demand = cust['Avg Monthly Demand (box)']
    
    # Hitung jarak ke semua DC
    dc_distances = []
    for _, dc in df_dc_deliver.iterrows():
        dc_coord = (dc['Latitude'], dc['Longitude'])
        dist = geodesic(cust_coord, dc_coord).km
        dc_distances.append((dc['DC ID'], dist, dc['Type']))
    
    # Urutkan berdasarkan Jarak dan Preferensi Type
    dc_distances.sort(key=lambda x: (x[1], {'Direct DC': 0, 'Direct CPU': 0, 'Indirect DC': 1, 'Indirect CPU': 1, 'Depo': 2}.get(x[2], 3)))
    
    # Alokasikan ke DC yang kapasitasnya cukup dan jaraknya wajar
    assigned = False
    for dc_id, dist, dc_type in dc_distances:
        if dist > MAX_DELIVERY_DISTANCE:
            continue  # Skip terlalu jauh
        
        dc_index = df_dc_deliver[df_dc_deliver['DC ID'] == dc_id].index[0]
        remaining = df_dc_deliver.at[dc_index, 'Remaining Capacity']
        
        if remaining >= cust_demand:
            df_dc_deliver.at[dc_index, 'Remaining Capacity'] -= cust_demand
            allocation_result.append({
                'Customer ID': cust['Customer ID'],
                'Latitude': cust['Latitude'],
                'Longitude': cust['Longitude'],
                'Nearest DC ID': dc_id,
                'Distance to DC (km)': round(dist, 2),
                'Avg Monthly Demand (box)': cust_demand
            })
            assigned = True
            break

    if not assigned:
        allocation_result.append({
            'Customer ID': cust['Customer ID'],
            'Latitude': cust['Latitude'],
            'Longitude': cust['Longitude'],
            'Nearest DC ID': 'UNASSIGNED',
            'Distance to DC (km)': None,
            'Avg Monthly Demand (box)': cust_demand
        })

# Step 4: Buat DataFrame hasil
df_customer_assigned = pd.DataFrame(allocation_result)

# Step 5: Total demand per DC
df_demand_per_dc = df_customer_assigned[df_customer_assigned['Nearest DC ID'] != 'UNASSIGNED'] \
    .groupby('Nearest DC ID')['Avg Monthly Demand (box)'].sum().reset_index()
df_demand_per_dc.columns = ['DC ID', 'Total Demand (box)']

# Step 6: Save ke CSV
df_demand_per_dc.to_csv('demand_per_dc.csv', index=False)
df_customer_assigned.to_csv('customer_assigned_to_dc.csv', index=False)

# Step 7: Pemetaan Region/Area
df_customer_assigned_with_area = df_customer_assigned.merge(
    df_dc_deliver[['DC ID', 'Distributor Area', 'REGION']],
    left_on='Nearest DC ID',
    right_on='DC ID',
    how='left'
)

df_demand_per_area = df_customer_assigned_with_area[df_customer_assigned_with_area['Nearest DC ID'] != 'UNASSIGNED'] \
    .groupby('REGION')['Avg Monthly Demand (box)'].sum().reset_index()

# Output akhir
print(df_demand_per_area)
print("✅ Hasil demand per DC disimpan ke 'demand_per_dc.csv'")
print("✅ Pemetaan customer disimpan ke 'customer_assigned_to_dc.csv'")


In [22]:
import pandas as pd

# Load data hasil assign sebelumnya
df_assign = pd.read_csv("customer_assigned_to_dc_no_capacity.csv")

# Hitung total demand per DC
df_demand_dc = df_assign.groupby('Nearest DC ID')['Avg Monthly Demand (box)'].sum().reset_index()
df_demand_dc = df_demand_dc.rename(columns={'Avg Monthly Demand (box)': 'total_demand_box'})
print(df_demand_dc)

df_assign.head(5)


   Nearest DC ID  total_demand_box
0          DC001       2617.487149
1          DC002       1451.503244
2          DC003       6052.556426
3          DC004        839.895140
4          DC005       3655.658944
5          DC006       1714.526280
6          DC007       1953.406371
7          DC008       1222.544183
8          DC009       2275.149136
9          DC010      10024.280811
10         DC011       2237.882598
11         DC012       1988.736534
12         DC013       3341.250311
13         DC014       3266.124029
14         DC015       3936.666527
15         DC016       2394.504440
16         DC017       5495.424580
17         DC018       1493.868871
18         DC019       1161.299071
19         DC020       2162.941119
20         DC021       3250.390567
21         DC022       1812.250785
22         DC023       3267.385348
23         DC024       3191.502889
24         DC025        989.383723
25         DC026       2180.274900
26         DC027       1743.310341
27         DC028    

,Customer ID,Latitude,Longitude,Nearest DC ID,Distance to DC (km),Avg Monthly Demand (box)
0,CUST000001,4.145344,117.912634,DC045,96.23,47.836860
1,CUST000002,4.145344,117.912634,DC045,96.23,10.774181
2,CUST000003,4.144048,117.912500,DC045,96.10,5.733333
3,CUST000004,4.144048,117.912500,DC045,96.10,1.541669
4,CUST000005,4.143269,117.654593,DC045,88.90,12.515233


In [24]:
df_demand_dc.head(5)

,Nearest DC ID,total_demand_box
0,DC001,2617.487149
1,DC002,1451.503244
2,DC003,6052.556426
3,DC004,839.895140
4,DC005,3655.658944


In [27]:
file_path = '1742574481558_Dataset-ME-2025.xlsx'
df_dc_final = pd.read_excel(file_path, sheet_name='DC Final', header=3)
df_dc = pd.read_excel(file_path, sheet_name='DC', header=3)
df_dist_center = pd.read_excel(file_path, sheet_name='Dist Center', header=3)

In [33]:
# --- DC Final ---
df_dc_final_map = pd.DataFrame({
    'DC Code': [df_dc_final.iloc[0, 1]],
    'City': [df_dc_final.iloc[0, 0]],
    'Latitude': [df_dc_final.iloc[0, 8]],
    'Longitude': [df_dc_final.iloc[0, 9]]
})

# --- DC ---
df_dc_map = pd.DataFrame({
    'DC Code': [df_dc.iloc[0, 1]],
    'City': [df_dc.iloc[0, 0]],
    'Latitude': [df_dc.iloc[0, 8]],
    'Longitude': [df_dc.iloc[0, 9]]
})

# --- Dist Center ---
df_dist_map = pd.DataFrame({
    'DC Code': [df_dist_center.iloc[0, 0]],
    'City': [df_dist_center.iloc[0, 3]],
    'Latitude': [df_dist_center.iloc[0, 7]],
    'Longitude': [df_dist_center.iloc[0, 8]]
})

# Gabungkan semua lokasi DC
df_all_dc_map = pd.concat([df_dc_final_map, df_dc_map, df_dist_map], ignore_index=True)

# Pastikan semua kolom tipe data yang benar
df_all_dc_map['Latitude'] = pd.to_numeric(df_all_dc_map['Latitude'], errors='coerce')
df_all_dc_map['Longitude'] = pd.to_numeric(df_all_dc_map['Longitude'], errors='coerce')

print(df_all_dc_map)


  DC Code         City   Latitude   Longitude
0   DC004        Barru  -4.405620  119.618770
1   DC038  Banjarmasin  -3.330249  114.595582
2   DC050       Kupang -10.180170  123.549420


In [30]:
# Temukan kolom yang mengandung kata 'city', 'latitude', dll
print([col for col in df_dc_final.columns if 'city' in col.lower()])
print([col for col in df_dc_final.columns if 'lat' in col.lower()])
print([col for col in df_dc_final.columns if 'dc' in col.lower()])


AttributeError: 'float' object has no attribute 'lower'

In [28]:
df_dc_final_map = df_dc_final[['DC Code', 'City', 'Latitude', 'Longitude']].copy()
df_dc_map = df_dc[['DC Code', 'City', 'Latitude', 'Longitude']].copy()
df_dist_map = df_dist_center[['DC Code', 'City', 'Latitude', 'Longitude']].copy()

df_dc_final_map.columns = ['dc_code', 'branch', 'lat', 'lon']
df_dc_map.columns = ['dc_code', 'branch', 'lat', 'lon']
df_dist_map.columns = ['dc_code', 'branch', 'lat', 'lon']

KeyError: "None of [Index(['DC Code', 'City', 'Latitude', 'Longitude'], dtype='object')] are in the [columns]"

In [26]:
# 1. Rename kolom agar konsisten
df_demand_dc = df_demand_dc.rename(columns={'Nearest DC ID': 'assigned_dc'})

# 2. Tambah kolom total_demand_cbm (jika belum)
box_to_cbm = 0.075
df_demand_dc['total_demand_cbm'] = df_demand_dc['total_demand_box'] * box_to_cbm

# 3. Merge dengan df_summary (kapasitas kendaraan per DC)
df_allocation_check = df_demand_dc.merge(df_summary, left_on='assigned_dc', right_on='Branch', how='left')

# 4. Hitung utilisasi kapasitas
df_allocation_check['utilization_ratio'] = (
    df_allocation_check['total_demand_cbm'] / df_allocation_check['origin_total_capacity_cbm']
).round(2)

# 5. Tampilkan hasil
df_allocation_check.sample(10)

,assigned_dc,total_demand_box,total_demand_cbm,Branch,origin_total_vehicle,origin_total_capacity_cbm,origin_total_petrol_per_100km,origin_avg_petrol_per_100km,utilization_ratio
15,DC016,2394.504440,179.587833,NaN,NaN,NaN,NaN,NaN,NaN
54,DC055,4905.002755,367.875207,NaN,NaN,NaN,NaN,NaN,NaN
33,DC034,7533.841607,565.038121,NaN,NaN,NaN,NaN,NaN,NaN
35,DC036,6917.255792,518.794184,NaN,NaN,NaN,NaN,NaN,NaN
6,DC007,1953.406371,146.505478,NaN,NaN,NaN,NaN,NaN,NaN
25,DC026,2180.274900,163.520617,NaN,NaN,NaN,NaN,NaN,NaN
41,DC042,7500.777323,562.558299,NaN,NaN,NaN,NaN,NaN,NaN
20,DC021,3250.390567,243.779293,NaN,NaN,NaN,NaN,NaN,NaN
11,DC012,1988.736534,149.155240,NaN,NaN,NaN,NaN,NaN,NaN
22,DC023,3267.385348,245.053901,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
df_edges.head(10)

NameError: name 'df_edges' is not defined